Youtube video clip of the senate hearing mentioning 85% of child deaths due to influenza were children who did not receive a flu vaccine: https://www.youtube.com/shorts/haluXyj40dk 

Senator Michael Bennet questioned Kennedy about changes to vaccine recommendations, pointing out that 2025 saw the highest number of childhood flu deaths in modern American history.

Criteria:
1. Your project has a recognizable and reproducible “data analytics workflow.” [Example: First the data is acquired, then necessary transformations and clean-up are performed, then the analysis and presentation work is performed]
2. Project includes data from at least two different types of data sources (e.g., two or more of these: (1) relational, (2) scraped web page, (3) web API.
3. Project includes at least one data transformation operation. [Examples: transforming from wide to long; converting columns to date format]
4. Project includes at least one grouping or aggregation.
5. Project includes at least one statistical analysis and at least one graphics that describes or validates your data.
6. Project includes at least one graphic that supports your conclusion(s).
7. Project includes at least one statistical analysis that supports your conclusion(s).
8. Project includes at least one feature that we did not cover in class! There are many examples: “I used Bokeh; I created a decision tree; I ranked the results; I created my presentation slides directly from my Jupyter Notebook; I figured out to use OAuth 2.0…”
9. Presentation. Was the presentation delivered in the allotted time (3 to 5 minutes)?
10. Presentation. Did you show (at least) one challenge you encountered in code and/or data, and what you did when you encountered that challenge? If you didn’t encounter any challenges, your assignment was clearly too easy for you!
11. Presentation. Did the audience come away with a clear understanding of your motivation for undertaking the project?
12. Presentation. Did the audience come away with a clear understanding of at least one insight you gained or conclusion you reached or hypothesis you “confirmed” (rejected or failed to reject…)?
13. Code and data. Have you delivered the submitted code and data where it is reproducible and self-contained—preferably in a Jupyter Notebook on GitHub? Am I able to fully reproduce your results with what you’ve delivered? You won’t receive full credit if your code references data on your local machine!
14. Code and data. Does all of the delivered code run without errors?
15. Deadline management. Were your draft project proposal, project, and presentation delivered on time? Any part of the project that is turned in late will receive a maximum grade of 80%. Please turn in your work on time! You are of course welcome to deliver ahead of schedule!

Grab data from the U.S. Centers for Disease Control data API.

In [ ]:
import pandas as pd
from sodapy import Socrata
import os
from dotenv import load_dotenv

# dependencies for web scraping
from bs4 import BeautifulSoup
import time
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

load_dotenv(override=True) 
MY_APP_TOKEN = os.getenv("API_KEY")

CDC_DATASET_ID = "vh55-3he6"

# Use the provided app token to bypass basic throttling
client = Socrata("data.cdc.gov", MY_APP_TOKEN)

# note 2009 did have mixed data for the Influenza A (H1N1) 2009 Monovalent vaccine
query = """
SELECT 
    *
WHERE 
    geography = 'United States'
    AND vaccine = 'Seasonal Influenza'
    AND dimension_type = 'Age'
    AND dimension = '6 Months - 17 Years'
LIMIT 50000
"""

try:
    results = client.get(CDC_DATASET_ID, query=query)
    cdc_api_df = pd.DataFrame.from_records(results)
           
    print(f"Success! Retrieved {len(cdc_api_df)} records.")
    
    
except Exception as e:
    print(f"Query Failed: {e}")

Success! Retrieved 173 records.



Using selenium to launch a background browser to render the webpages since the CDC FluView site uses client side rendering to populate the data on the page. Once the page has been loaded and populated in the background browser, then BeautifulSoup is used to scrape the weekly hospitalization rate and pediatric deaths. 

Note: This is not an optimal solution since the data is also included in a downloadable csv, this is designed to demonstrate data gathering using web scraping. FluView dashboards start in October 2024. Also 2026-week-13 on the CDC site has an issue retreiving data from its backend, so unfortunately that data is not available. 

FluView Homepage: https://www.cdc.gov/fluview/index.html 
Disease burden data: https://www.cdc.gov/flu-burden/php/data-vis/index.html

In [ ]:
def scrape_fluview_structural(start_year, start_week, end_year, end_week):
    """
        Scrapes CDC FluView dashboard pages by targeting section headers and then 
        extracting data from their respective content blocks.
    """
    all_data = []

    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")

    print("Launching background browser...")
    driver = webdriver.Chrome(options=chrome_options)

    for year in range(start_year, end_year + 1):
        for week in range(1, 54):

            # Only pull data for weeks between the start year/week and end year/week
            if year == start_year and week < start_week:
                continue
            if year == end_year and week > end_week:
                break

            week_str = f"{week:02d}"
            url = f"https://www.cdc.gov/fluview/surveillance/{year}-week-{week_str}.html"

            try:
                print(f"Loading: {year} Week {week_str}...")
                hosp_rate = None
                pediatric_deaths = None

                driver.get(url)
                time.sleep(10)  # need a buffer for the client side data to load

                soup = BeautifulSoup(driver.page_source, "html.parser")

                # parse the layout by div sections
                cards = soup.find_all(
                    "div", class_="cove-visualization__inner"
                )

                if not cards:
                    print(f"-> No report elements found for {year}-Wk{week_str}")
                    continue

                
                for card in cards:
                    # Find the section header
                    header_tag = card.find(["h2", "header"])
                    if not header_tag:
                        continue

                    header_text = header_tag.get_text(strip=True).lower()

                    # 1. Find the hospitilization rate under the FluSurv-NET header
                    if "nhsn hospital respiratory data" in header_text:
                        content_block = card.find("div", class_="cove-prose")
                        if content_block:
                            card_text = content_block.get_text(
                                separator=" ", strip=True
                            ).replace("\xa0", " ")
                            rate_match = re.search(r"([\d.]+)", card_text)
                            if rate_match:
                                hosp_rate = float(rate_match.group(1))

                    # 2. Find the Pediatric Deaths header and associated data
                    if "pediatric deaths" in header_text:
                        # First, look for the large callout number span (1.5rem)
                        large_num_span = card.find("span", style=re.compile(r"size:\s*1\.5rem"))

                        if large_num_span and large_num_span.get_text(strip=True):
                            # If the large span has text, extract that number
                            num_text = large_num_span.get_text(strip=True)
                            death_match = re.search(r"([\d,]+)", num_text)
                            if death_match:
                                pediatric_deaths = int(death_match.group(1).replace(",", ""))
                        else:
                            # If the span is empty or missing, check the block text
                            # The number must appear right before 'influenza-associated'
                            content_block = card.find("div", class_="cove-prose")
                            if content_block:
                                card_text = content_block.get_text(
                                    separator=" ", strip=True
                                ).replace("\xa0", " ")
                                strict_match = re.search(
                                    r"([\d,]+)\s+influenza-associated", card_text
                                )

                                if strict_match:
                                    pediatric_deaths = int(
                                        strict_match.group(1).replace(",", "")
                                    )
                                else:
                                    pediatric_deaths = 0


                all_data.append(
                    {
                        "year": year,
                        "week": week,
                        "hospitalization_rate": hosp_rate,
                        "pediatric_deaths": pediatric_deaths,
                    }
                )

                print(
                    f"   Success -> Rate: {hosp_rate} | Deaths: {pediatric_deaths}"
                )

            except Exception as e:
                print(f"   Error parsing {year}-Wk{week_str}: {e}")
                continue

    driver.quit()

    # extract to a pandas dataframe
    df = pd.DataFrame(
        all_data,
        columns=["year", "week", "hospitalization_rate", "pediatric_deaths"],
    )
    return df


# run the scraping function
if __name__ == "__main__":
    flu_scraped_df = scrape_fluview_structural(
        start_year=2024, start_week=36, end_year=2026, end_week=17
    )

    print("\nExtraction finished. Final Pandas Dataframe view:")
    print(flu_scraped_df.head())
    flu_scraped_df.to_excel('scraped_data.xlsx', index=False)

Launching background browser...
Loading: 2024 Week 36...
   Success -> Rate: None | Deaths: 3
Loading: 2024 Week 37...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 38...
   Success -> Rate: None | Deaths: 1
Loading: 2024 Week 39...
   Success -> Rate: None | Deaths: 1
Loading: 2024 Week 40...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 41...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 42...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 43...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 44...
   Success -> Rate: None | Deaths: 1
Loading: 2024 Week 45...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 46...
   Success -> Rate: None | Deaths: 1
Loading: 2024 Week 47...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 48...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 49...
   Success -> Rate: None | Deaths: 0
Loading: 2024 Week 50...
   Success -> Rate: None | Deaths: 2
Loading: 2024 Week 51...
   Success ->

Note: June is omitted from each year since the CDC considers a flu season to be from July to May of the following year. 
Clean up data and transform as needed.

In [ ]:
# convert_year season to just year by keeping the first 4 characters
cdc_api_df['year_season'] = cdc_api_df['year_season'].astype(str).str[:4]
# Ensure numeric columns are actually numbers
numeric_cols = ['coverage_estimate', 'population_sample_size', 'year_season', 'month']
cdc_api_df[numeric_cols] = cdc_api_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
# Sort by year_season and month in ascending order
cdc_api_df = cdc_api_df.sort_values(by=['year_season', 'month']).reset_index(drop=True)



# print(cdc_api_df.head())
cdc_api_df.to_excel('output.xlsx', index=False)